# Quantum Error Correction workflows

Quantum error correction (QEC) is a major area of active research in quantum computing as we transition from the noisy intermediate-scale quantum (NISQ) era to the fault-tolerant (FT) quantum era.

After working through this part of the tutorial, you should be able to:
- Define error correction and some of the basic vocabulary of QEC 
- Implement experiments on IQM quantum hardware that demonstrates the effect of noise on physical qubits
- Implement simple QEC experiments on IQM quantum hardware
- Explain why QEC will eventually require tight QPU and HPC integration 
- Use CUDA-QX QEC to construct and simulate experiments beyond the scope of current QPUs, including using GPU-accelerated decoding

# Part 0 - Environment setup

If you are using Google Colab+IQM Resonance, make sure you have an API token for IQM Resonance, and run the cell below to install packages you need.

In [ ]:
import sys

# Only install packages when running on Google Colab.
# On the private SLURM system the environment is managed by uv and packages are already present.
if "google.colab" in sys.modules:
    import subprocess
    def pip_install(*packages):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    pip_install("cudaq", "iqm-cortex-cli","matplotlib", "numpy","cudaq-qec")
    print("✅ Installation complete. If the runtime restarted, continue from the next cell.")

Let's take care of our imports as well.

In [44]:
import cudaq
import os
import cudaq_qec as qec

import numpy as np

Now let's connect to the IQM device.

In [ ]:
if "google.colab" in sys.modules:
    # Run on IQM Resonance
    if not os.environ.get("IQM_TOKEN"):
        os.environ["IQM_TOKEN"] = input("Input your API token")
    quantum_computer = "emerald"
    if not os.environ.get("IQM_SERVER_URL"):
        os.environ["IQM_SERVER_URL"] =  f"https://cocos.resonance.meetiqm.com/{quantum_computer}"
    cudaq.set_target("iqm")

else:
    # Run on LRZ
    IQM_SERVER_URL = os.environ.get("IQM_SERVER_URL", "https://qan06.beast.lrz.de/station/")
    IQM_TOKEN_FILE = os.path.expanduser("~/.iqmclient_eqe1_token")
    if not os.environ.get("IQM_TOKEN"):
        with open(IQM_TOKEN_FILE) as f:
            os.environ["IQM_TOKEN"] = f.read().strip()
    cudaq.set_target("iqm", url=IQM_SERVER_URL)

if cudaq.get_target().name=="iqm":
    print("✅ Connected to IQM target")
else:
    print("Failed to connect to IQM device. Check your environment setup.")

✅ Connected to IQM target


# Part 1 - Basic concepts of error correction

## 1.1 - What's the problem?

Meet A and B. A and B live far away from each other, but they want to cooperate.

A has a qubit that she wants to send B. The problem is that their communication is garbled - there's a probability $p$ that any qubit she sends to Bob gets flipped. For now, we're going to assume this is the only possible error. An error like this is called a **bit-flip error**, and the presence of noise means they are sending information across a **noisy channel**.

So what happens when A sends B a message? 

<img src="AAndBnEquals1.jpg" alt="Diagram of outcomes when A sends B a 0." width="500px">

Well, let's say B measures $\left|0\right\rangle$. B doesn't know what A sent, but as long as $p<1/2$, B reasons that it's more likely that A sent a $\left|0\right\rangle$ and there was no error. There's technically some Bayesian probability analysis hiding here, but it's straightforward to check that if A is sending random bits, B will guess right with probability $1-p$ and guess wrong with probability $p$.

So with probability $p$, B gets the wrong message **and has no idea that the message is wrong**. This is called a **logical error**, and this will destroy B's future computations. (Imagine, for example, that he feeds this qubit into a conditional statement - then we're going to get garbage out since the condition is totally wrong!)

Let's try this experiment - we can compare simulation with actual quantum hardware.

> **TASK**: Write the missing function below.

In [29]:
@cudaq.kernel
def prepare_state_z(data_qubits: cudaq.qview, logical_state:bool):
    '''
    Input: logical_state. Boolean, where False represents |0> and True represents |1>
    Output: data_qubits. 1-qubit qvector of physical qubits, prepared in the logical state |logical_state>
    '''
    
    if logical_state:
        x(data_qubits)

@cudaq.kernel
def qubit_experiment(logical_state:bool):
    '''
    Input: logical_state: Boolean representing whether we are encoding a logical 0 or logical 1
    Effect: Encodes logical state into physical qubits, then measures those qubits.
    '''
    data_qubits = cudaq.qvector(1)
    prepare_state_z(data_qubits,logical_state) #A prepares her qubit
    pass #A sends her qubit to B
    mz(data_qubits) # B measures his qubit

# Some testing:

# Simulate the state preparation quantum circuit
cudaq.set_target("qpp-cpu")
sim_result = cudaq.sample(qubit_experiment,False)
print(f"Let's see what sampling our qvector does in simulation: {sim_result}")

shots=1000
cudaq.set_target("iqm",url=iqm_url)
# Actually run the quantum circuit we're creating on hardware so that we can see what CUDA-Q spits out for us: 
hw_result = cudaq.sample(qubit_experiment,False, shots_count=shots)
print(f"And running it on hardware: {hw_result}")

Let's see what sampling our qvector does in simulation: { 0:1000 }

And running it on hardware: { 0:1000 }



Now we can calculate the error rate. (This will look a little silly, but bear with us!)

In [30]:
def get_error_rate(results:dict,logical_state:bool):
    '''
    Input:  results: dictionary representing measurement results
            logical_state: Boolean representing whether we are encoding a logical 0 or logical 1
    Output: Error rate as a percentage
    '''
    shots = sum(results.values()) #count the total shots
    if logical_state:
        errors = 1-results["1"]/shots #this is basically results["0"]/shots, but if there are no 0 counts, results["0"] would raise a ValueError.
    else:
        errors = 1-results["0"]/shots

    return 100*errors # return error rate as a percent

Now let's run the experiment end to end an dcompare how it goes in simulation and on hardware:

In [31]:
shots=1000
logical_state = False

cudaq.set_target("qpp-cpu")
sim_results = cudaq.sample(qubit_experiment,logical_state,shots_count=shots)
sim_error_rate = get_error_rate(sim_results,logical_state)
print(f"Simulation: error rate is {sim_error_rate}%")

#Hardware
cudaq.set_target("iqm",url=iqm_url)
hardware_results = cudaq.sample(qubit_experiment,logical_state,shots_count=shots)
hardware_error_rate = get_error_rate(hardware_results,logical_state)
print(f"Hardware: error rate is {hardware_error_rate}%")


Simulation: error rate is 0.0%
Hardware: error rate is 0.10000000000000009%


Not surprisingly, simulation has no errors, but hardware, of course, does (unless you got lucky). This error rate should be pretty low, but remember, we haven't even done anything to our qubit!

>**TASK**: Make sure you try the experiment above with both possible states. Do you notice a difference?

>**TASK**: Imagine you had this sort of error rate every time you applied a quantum gate. How many gates could you apply and still get the right result 90% of the time?

In [15]:
desired_correctness = 0.9
error_rate = 0.004
# desired_correctness = (1-error rate)^(number of gates). So
num_gates = np.log(desired_correctness)/np.log(1-error_rate)
print(num_gates)

26.287413466036956


To contextualize the last task, running Shor's algorithm to break RSA encryption is estimated to take $\sim 10^{12}$ gates. This... is going to be a problem.

Let's see if we can do better.

## 1.2 - A simple error correcting code

We're going to introduce some language: A's intended message will be called a **logical qubit**, which we'll denote with a subscript $L$. So imagine A wanted to send the logical qubit $\left|0\right\rangle_L$. What she's going to do is to send B $n$ qubits, called **physical qubits**. A and B agree beforehand that when A is sending a logical $\left|0\right\rangle_L$, she **encodes** that message by sending $n$ physical $\left|0\right\rangle$ qubits, and she encodes $\left|1\right\rangle_L$ as $n$ physical $\left|1\right\rangle$ bits.

How does this help? Let's look at $n=3$. First imagine B recieves the message $\left|000\right\rangle$. What does B think? 

<img src="AAndB000.jpg" alt="Thought process when B recieves 000 using the n=3 repetition code." width="500px">

B knows that A either sent $\left|000\right\rangle$ or $\left|111\right\rangle$. Given that he received $\left|000\right\rangle$, either zero or three errors occurred. Which is more likely? If $p<1/2$, *any* outcomes with more errors are less common. So B will reason that the most likely situation is that A sent $\left|000\right\rangle$ and meant logical 0.

But what happens if B recieves the message, say, $\left|010\right\rangle$? 

<img src="AAndB010.jpg" alt="Thought process when B recieves 010 using the n=3 repetition code." width="500px">

That wasn't part of the agreement, so an error must have happened! B can't know for sure what A intended. But B can make a good guess. Since one error is more likely than two, that it's more likely that A sent $\left|000\right\rangle$ and there was an error in the middle qubit than A sent $\left|111\right\rangle$ and there were errors in the other two qubits.

Extrapolating, B is going to make a guess for A's message, that is, **decode** A's message, like this:

|**Physical measurement**|**Guess for logical qubit**|
|:---:|:---:|
|More 0s than 1s| $\left\lvert 0\right\rangle_L$|
|More 1s than 0s| $\left \lvert 1\right\rangle_L$|
|Equal number (if $n$ is even)| Unknown, but an error has happened|

This doesn't mean that B is guaranteed success. But he's **more likely** to be successful than in our previous example.

Let's try it out. Let's repeat the same experiment as before, but A now needs to prepare the **encoded** state.

> **TASK**: Write the missing function below.

In [32]:
@cudaq.kernel
def prepare_state_repetition_z(data_qubits:cudaq.qview,logical_state:bool):
    '''
    Inputs: data_qubits: n-qubit qvector, assumed to be in the all 0 state
            logical_state: Boolean, where False represents |0>_L and True represents |1>_L
    Effect: Prepares data_qubits in the logical state |logical_state>
    '''
    
    if logical_state:
        x(data_qubits)

@cudaq.kernel
def qubit_experiment_repetition_z(logical_state:bool,n:int):
    '''
    Inputs: logical_state: Boolean, where False represents |0>_L and True represents |1>_L
            n: Integer, number of physical qubits in the encoding
    Effect: Encodes logical state into physical qubits, then measures those qubits.
    '''
    data_qubits = cudaq.qvector(n)
    prepare_state_repetition_z(data_qubits,logical_state) #A prepares her qubits
    pass #A sends her qubits to B
    mz(data_qubits) # B measures his qubits

Now we need to prepare B's decoder. Remember:

- If we measure a state with more physical 0s then 1s, interpret that as a logical 0.
- If we measure a state with more physical 1s then 0s, interpret that as a logical 1.
- If we measure an equal number of 0s and 1s, then we know errors occurred, but we don't know what the logical bit was, so there's not a good strategy to deal with this case.

> **TASK**: Write the decoder function below.

In [33]:
def decode_repetition(results,target_logical,n):
    '''
    Inputs: results: results dictionary from get_measurement call
            target_logical: Boolean representing the intended logical state (to determine what's an error)
            n: Integer, number of physical qubits in the encoding
    Outputs: physical_errors: number of counts in results with at least one physical error
             logical_errors: number of counts in results with a logical error
             ambiguous errors: number of counts in results with an ambiguous result (exactly n/2 errors)
             correct: number of counts in resultswhich decoded correctly
    '''
    
    correct_logical = "1" if target_logical else "0"
    correct=0
    logical_errors=0
    ambiguous_errors=0
    physical_errors=0
    for bitstring, counts in results.items():
        unflipped_bits = bitstring.count(correct_logical)
        if unflipped_bits != n:
            physical_errors += counts
        if unflipped_bits > n/2:
            correct += counts
        elif unflipped_bits < n/2:
            logical_errors += counts
        else:
            ambiguous_errors += counts
    return physical_errors, logical_errors, ambiguous_errors, correct

Now we can run the experiment.

In [34]:
shots=1000
logical_state = False
n=3

#Simulation

cudaq.set_target("qpp-cpu")

sim_results = cudaq.sample(qubit_experiment_repetition_z,logical_state,n,shots_count=shots)
sim_physical_errors, sim_logical_errors, sim_detected_errors, sim_correct = decode_repetition(sim_results,logical_state,n)
print(f"Simulation: physical error rate is {100*sim_physical_errors/shots}%")
print(f"Simulation: logical error rate is {100*sim_logical_errors/shots}%")
print(f"Simulation: detected error rate is {100*sim_detected_errors/shots}%")

#Hardware
cudaq.set_target("iqm")
hardware_results = cudaq.sample(qubit_experiment_repetition_z,logical_state,n,shots_count=shots)
hw_physical_errors, hw_logical_errors,hw_ambiguous_errors,hw_correct = decode_repetition(hardware_results,logical_state,n)
print(f"Hardware: physical error rate is {100*hw_physical_errors/shots}%")
print(f"Hardware: logical error rate is {100*hw_logical_errors/shots}%")
print(f"Hardware: ambiguous error rate is {100*hw_ambiguous_errors/shots}%")


Simulation: physical error rate is 0.0%
Simulation: logical error rate is 0.0%
Simulation: detected error rate is 0.0%
Hardware: physical error rate is 0.4%
Hardware: logical error rate is 0.0%
Hardware: ambiguous error rate is 0.0%


> TASK: Experiment with larger values of $n$ to see if you can improve your results! And experiment with both $\left|0\right\rangle_L$ and $\left|1\right\rangle_L$.

## Part 2 - Why aren't we just done?

It seems like everything is going well! We've figured out a way to deal with errors! So why isn't this the whole story? The answer is that so far, we've been ignoring the "qu" in "qubit."

To see the issue, what happens if A wants to send B the message 
$$\left|+\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|0\right\rangle_L+\left|1\right\rangle_L\right)?$$
Then she needs to send B the physical state
$$\left|+\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|000\right\rangle+\left|111\right\rangle\right)$$

Likewise, if A wants to send B the message 
$$\left|-\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|0\right\rangle_L-\left|1\right\rangle_L\right)$$
she needs to send B the physical state
$$\left|-\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|000\right\rangle-\left|111\right\rangle\right)$$

What happens once B measures his qubits to check if an error has occurred? Let's try repeating our previous experiment but for the logical states $\left|+\right\rangle$ or $\left|-\right\rangle$.

> **TASK**: Write the missing function below.

In [35]:
@cudaq.kernel
def prepare_state_repetition_pm(data_qubits:cudaq.qview,logical_state:bool):
    '''
    Input:  data_qubits: n-qubit qvector of physical qubits,assumed to be in the all-0 state
            logical_state. Boolean, where False represents |+>_L and True represents |->_L
    Effect: Prepare data_qubits in the logical state |logical_state>_L
    '''

    #Prepare the state |000...> + |111...>
    h(data_qubits[0])
    for i in range(1,len(data_qubits)):
        x.ctrl(data_qubits[0],data_qubits[i])

    # Change the sign of the |111...> state if logical_state is True
    if logical_state:
        z(data_qubits[0])

@cudaq.kernel
def qubit_experiment_repetition_pm(logical_state:bool,n:int):
    '''
    Inputs: logical_state: Boolean representing whether we are encoding a logical + or logical -
            n: int, number of qubits in the encoding
    Effect: Encodes logical state into physical qubits, then measures those qubits.
    '''
    data_qubits = cudaq.qvector(n)
    prepare_state_repetition_pm(data_qubits,logical_state) #A prepares her qubits
    pass #A sends her qubits to B
    mz(data_qubits) # B measures his qubits

Now let's see what the output of our circuit is. We won't bother decoding, for reasons you'll see shortly. Importantly, make sure you try both the logical states $\left|+\right\rangle$ and $\left|-\right\rangle$:

In [36]:
shots=10000 # let's run lots of shots to get really good statistics
n=3

cudaq.set_target("qpp-cpu")
plus_state = cudaq.sample(qubit_experiment_repetition_pm,False,n,shots_count=shots)
minus_state = cudaq.sample(qubit_experiment_repetition_pm,True,n,shots_count=shots)

print(f"Simulation, target state +: {plus_state}")
print(f"Simulation, target state -: {minus_state}")

Simulation, target state +: { 000:5041 111:4959 }

Simulation, target state -: { 000:4921 111:5079 }



Now B can't tell the difference between these two logical states $\left|+\right\rangle_L$ and $\left|-\right\rangle_L$, no matter how he decodes! Notice also that if only a single physical qubit has an error of the form $\left|0\right\rangle \rightarrow \left|0\right\rangle$, $\left|1\right\rangle \rightarrow -\left|1\right\rangle$, B would have a **logical error** even if he could check which state he had (because this **phase-flip error** turns $\left|+\right\rangle_L$ into $\left|-\right\rangle_L$).

Now, it's true that if B were to measure, say $\left|010\right\rangle$, he would know that an error occurred. But then what has happened? The state A sent B is now **gone** because the measurement collapsed the superposition. So B can't even do any processing on that state to try to fix the error!

Because of these two problems, quantum error correction is a lot tricker than classical error correction.

## Part 3 - Why do we want QPU+HPC for QEC?

Right now we're doing the measurements and decoding afterward. But as we move towards fault-tolerant quantum computing, decoding is going to have to happen **in the middle of quantum computations**. This happens because it turns out that certain operations, if performed on *logical* qubits, require you to make an error-corrected measurement in order to determine what quantum operation to apply to your physical qubits.

Furthermore, we can't just let errors accumulate forever if we're running a long circuit - we have to detect them and correct them **as they happen.** 

So the workflow we need looks like this:

<img src="QECworkflow.jpg" alt="QEC workflow" width=500px>


# Part 4 (Optional) - Implementing a code that detects both bit-flip and phase-flip errors

We're now going to introduce a simple quantum code called the [[4,2,2]] code. It isn't a very good code, because it can only *detect* an error, not correct it. But it's useful because it illustrates the basic principles, and we can still use it to conduct a simple QEC experiment on IQM hardware.

## 4.1 - The [[4,2,2]] code

The [[4, 2, 2]] encodes **two** logical qubits into **four** physical qubits. The encoding looks like the following:

| Logical message | Physical qubits |
| :---------------: | :---------------: |
| $\left\lvert00\right\rangle_L$ | $\frac{1}{\sqrt{2}}\left(\left\lvert0000\right\rangle+\left\lvert1111\right\rangle\right)$ |
| $\left\lvert01\right\rangle_L$ | $\frac{1}{\sqrt{2}}\left(\left\lvert0011\right\rangle+\left\lvert1100\right\rangle\right)$ |
| $\left\lvert10\right\rangle_L$ | $\frac{1}{\sqrt{2}}\left(\left\lvert0101\right\rangle+\left\lvert1010\right\rangle\right)$ |
| $\left\lvert11\right\rangle_L$ | $\frac{1}{\sqrt{2}}\left(\left\lvert0110\right\rangle+\left\lvert1001\right\rangle\right)$ |

Let's write a function that prepares these states. The easiest way is to prepare $\left|00\right\rangle_L$ and then manipulate it.

> **TASK**: Write the missing function below.

In [37]:
@cudaq.kernel
def prepare_state(data_qubits:cudaq.qview, logical_state:list[bool]):
    '''
    Input: data_qubits. 4-qubit qvector of physical qubits, assumed in the all 0 state
            logical_state: List of two booleans representing the qubit states to encode
    Effect: Prepare data_qubits in the logical state |logical_state>
    '''

    # Prepare the state |00>_L
    h(data_qubits[0])
    for i in range(1,4):
        x.ctrl(data_qubits[0],data_qubits[i])
    
    # Flip the relevant qubits
    if logical_state[0]:
        x(data_qubits[0])
        x(data_qubits[2])
    if logical_state[1]:
        x(data_qubits[2])
        x(data_qubits[3])

> **TASK**: What tells us whether the first logical qubit is 0 or 1? What about the second logical qubit?

## 4.2 - Detecting errors with stabilizers

The next thing we need to do is to decide what to measure to check if there has been an error. Our goal is to measure some properties (actually quantum operators), called **stabilizers**, such that:

1) If an error has occurred, some measurement reads 1.
2) If no error has occurred, all of the measurements read 0.
3) Most importantly, if there was no error, the measurement process does not disturb the logical state. We don't want to collapse $\left|+\right\rangle_L$ into $\left|0\right\rangle_L$ or $\left|1\right\rangle_L$ just because we're checking for errors. 

A bad example would be to measure the first physical qubit. Let's see why.

Suppose the physical qubits are in the state $\left|00\right\rangle_L$ (with no errors), and we measure the first qubit. Then we have a 50% chance of collapsing the physical qubits into the state $\left|0000\right\rangle$, and a 50% chance of collapsing the physical qubits into the state $\left|1111\right\rangle$. Importantly, neither of these states are equal to $\left|00\right\rangle_L$! So we have destroyed the logical qubit. That's not good.

So, what could we measure? First off, we can't measure the actual data qubits. We need an extra **ancilla** qubit so that we don't immediately collapse the wavefunction of the data qubits. From there, properties (2) and (3) are our guide; we're looking for common features of **all** the logical states.

We will need two stabilizers, so we will need two ancillas. The idea is that measuring each ancilla tells us something about the data qubits, but not their full state.

For example, the first stabilizer is referred to as $ZZZZ$. This just measures the **parity** of how many physical qubits are in the state $\left|1\right\rangle$. This means that if an even number of physical qubits are in the state $\left|1\right\rangle$, then the ancilla should end up in the state $\left|0\right\rangle$. If an odd number of physical qubits are in the state $\left|1\right\rangle$, then the ancilla should end up in the state $\left|1\right\rangle$.

The other stabiilizer is referred to as $XXXX$. There are two (it turns out equivalent) ways to think about this measurement:
    1.If you flip all the physical qubits, does the overall state stay the same (in which case you measure 0) or pick up a minus sign (in which case you measure 1)? 
    2. It measures the **parity** of how many physical qubits are in the state $\left|-\right\rangle$. (We haven't written our logical states in the $+/-$ basis, but you can do so; you can check that all of them have an even number of $\left\lvert - \right\rangle$ qubits.)

Let's write a circuit to measure these stabilizers. The $ZZZZ$ stabilizer is a little simpler. Hint for the $XXXX$ stabilizer: remember that Hadamard gates swap the states $\left|0\right\rangle \leftrightarrow \left|+\right\rangle$ and $\left|1\right\rangle \leftrightarrow \left|-\right\rangle$.

> **TASK**: Convince yourself that the description of the stabilizers below are accurate. Then fill in the missing circuits.

In [38]:
@cudaq.kernel
def measure_stabilizers(data_qubits:cudaq.qview, ancilla_qubits:cudaq.qview):
    '''
    Inputs: data_qubits: qvector of 4 qubits representing physical qubits
            ancilla_qubits: qvector of 2 qubits used to measure stabilizers
    Effect: measures ancilla_qubits to give stabilizer results
    '''
    for qubit in data_qubits:
        # ZZZZ stabilizer:
        # For every data qubit, flip the first ancilla if the data qubit is a 1. Convince yourself that this flips the ancilla if and only if an odd number of data qubits are |1>.
        x.ctrl(qubit, ancilla_qubits[0])

        # XXXX stabilizer:
        # For every data qubit, flip the second ancilla if the data qubit is a -. Convince yourself that this flips the ancilla if and only if an odd number of the data qubits are |->.
        h(qubit)
        x.ctrl(qubit, ancilla_qubits[1])
        h(qubit)

    # Now measure the ancilla, but not the data qubits
    mz(ancilla_qubits)

## 4.3 - How do we use the stabilizers?

Imagine there's a bit-flip error on one of the qubits. What will happen? For example, if there is a bit-flip error in the third qubit,
$$\left|00\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|0000\right\rangle+\left|1111\right\rangle\right) \rightarrow \frac{1}{\sqrt{2}}\left(\left|0010\right\rangle+\left|1101\right\rangle\right)$$
When we measure $ZZZZ$, now we have an odd number of 1s! So $ZZZZ$ will measure 1 instead of 0, and we know there's an error. However, the $XXXX$ stabilizer won't change.

Likewise, if there's a phase-flip error on, for example, the second qubit,
$$\left|00\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|0000\right\rangle+\left|1111\right\rangle\right) \rightarrow \frac{1}{\sqrt{2}}\left(\left|0000\right\rangle-\left|1111\right\rangle\right)$$
Now when we flip all four qubits, the state picks up a minus sign! So $XXXX$ will measure 1 instead of 0, and we know there's an error. However, the $ZZZZ$ stabiilizer won't change.

So we see that this code can *detect* both a bit-flip and a phase-flip error. However, we don't know how to *correct* that error - you can convince yourself that flipping any of the four qubits will lead to the same result, so we don't know which qubit to flip back.

Worse, what happens if we have **two** errors? E.g. imagine there's a bit-flip error on both the third and fourth qubits:
$$\left|00\right\rangle_L = \frac{1}{\sqrt{2}}\left(\left|0000\right\rangle+\left|1111\right\rangle\right) \rightarrow \frac{1}{\sqrt{2}}\left(\left|0011\right\rangle+\left|1100\right\rangle\right) = \left|01\right\rangle_L$$

Uh oh! Now our message has been garbled (there has been a **logical error**), and we can't possibly know that this error occurred since the new state is a valid logical state!

Let's write our decoding function. Its job is to take the ancilla measurement **and** measurement of the data qubits and either declare that there was a physical error (so we should throw out this shot) or figure out what logical state the qubits should now be in.

In [39]:
def decode(bitstring):
    '''
    Input: bitstring: binary string whose first four bits are a measurement of the data qubits and last two bits are ancillas
    Outputs: error: Boolean, True if an error has been detected by the stabilizer measurements
             logical_state: list of two Booleans, logical state of the qubits (assuming no error has been detected)
    '''
    error = (bitstring[4:] != '00')
    first_bit = bitstring[0]!=bitstring[1] #remember the first logical qubit measures the parity of the first two (or last two) physical qubits
    second_bit = bitstring[0]!=bitstring[2] #remember the second logical qubit measures the parity of the first+third or second+fourth physical qubits

    return error, [first_bit, second_bit]    

> TASK: What happens if there is an error in the ancilla qubit? How does this affect our decoder?

## 4.4 - A (slightly simplified) quantum memory experiment

If we're going to do this on real hardware, we want to ask the following question: how often do logical errors occur without us detecting them? To find out, let's do the following (slightly simplified) "quantum memory experiment":

1) Prepare a target logical state.
2) Measure the stabilizers.
3) Measure the data qubits. (Notice: this destroys the logical state, but for this code, we can still tell the logical states apart based on this measurement result.)
4) Count the number of shots where the stabilizers both say 0 but the data qubits tell us the wrong logical state, i.e., we have an undetected logical error.

The fraction #4 divided by the total shots give us a good proxy for how well our code is working - the higher this number, the more often we are failing to detect errors.

In [40]:
@cudaq.kernel
def simplified_memory_expt(target_logical:list[bool]):
    '''
    Input: target_logical: list of two Booleans representing desired logical state
    Effect: Prepares 4 physical qubits in logical state target_logical, then measures stabilizers to check for errors 
    '''
    data = cudaq.qvector(4)
    prepare_state(data,target_logical)
    ancilla = cudaq.qvector(2)
    measure_stabilizers(data, ancilla)
    mz(data)

def process_experiment(target_logical:list[bool], measurements:dict):
    '''
    Inputs: target_logical: list of two Booleans representing the intended logical state
            measurements: Dictionary of measurement results from running a quantum circuit
    Outputs: logical_errors: float, percentage of shots with undetected errors (i.e. syndromes are both 0 but the qubits are not in the right logical state)
             detected_errors: float, percentage of shots which had a detected physical error
    '''
    logical_errors = 0
    detected_errors = 0

    shot_count=sum(measurements.values())

    for bitstring,shots in measurements.items():
        error, logical_measurements = decode(bitstring)
        if error:
            detected_errors+=shots
        elif logical_measurements != target_logical:
            logical_errors += shots
    
    return 100*logical_errors/shot_count, 100*detected_errors/shot_count

Now let's run the experiment! Starting with the simulator:

In [41]:
shots = 1000
logical_list = [[a, b] for a in [False, True] for b in [False,True]]

cudaq.set_target("qpp-cpu")

for logical in logical_list:
    results = cudaq.sample(simplified_memory_expt,logical,shots_count=shots)
    undetected_rate, detected_rate = process_experiment(logical,results)
    print("Sim: undetected logical errors for state ",  logical, ": ",undetected_rate,"%")
    print("Sim: detected errors for state ",  logical, ": ",detected_rate,"%")

Sim: undetected logical errors for state  [False, False] :  0.0 %
Sim: detected errors for state  [False, False] :  0.0 %
Sim: undetected logical errors for state  [False, True] :  0.0 %
Sim: detected errors for state  [False, True] :  0.0 %
Sim: undetected logical errors for state  [True, False] :  0.0 %
Sim: detected errors for state  [True, False] :  0.0 %
Sim: undetected logical errors for state  [True, True] :  0.0 %
Sim: detected errors for state  [True, True] :  0.0 %


> **TASK**: What do you observe in simulation? Is this what you expected?

Finally, let's try running this on real hardware:

In [42]:
cudaq.set_target("iqm")
for logical in logical_list:
    hw_results = cudaq.sample(simplified_memory_expt, logical, shots_count=shots)
    hw_undetected_rate, hw_detected_rate = process_experiment(logical,hw_results)
    print("Hardware: undetected logical errors for state ",  logical, ": ",hw_undetected_rate,"%")
    print("Hardware: detected errors for state ",  logical, ": ",hw_detected_rate,"%")

Hardware: undetected logical errors for state  [False, False] :  21.4 %
Hardware: detected errors for state  [False, False] :  56.1 %
Hardware: undetected logical errors for state  [False, True] :  13.7 %
Hardware: detected errors for state  [False, True] :  55.3 %
Hardware: undetected logical errors for state  [True, False] :  4.5 %
Hardware: detected errors for state  [True, False] :  93.8 %
Hardware: undetected logical errors for state  [True, True] :  3.1 %
Hardware: detected errors for state  [True, True] :  95.6 %


# Part 5 - For the future: CUDA-QX error correction tools

The [[4,4,2]] code only detects errors, just like our classical 2-bit repetition code. But we want to be able to both detect that an error has occurred and correct it - we have to switch to a different code, analogous to our longer classical repetition codes.

The most commonly used family of codes is called the (rotated) "surface code." It encodes one logical qubit into $d^2$ physical qubits, where $d$ is a parameter. For the surface code, $d$ happens to be equal to the *code distance*, which tells us that the surface code can correct approximately $d/2$ errors.

Importantly, the surface code requires $d^2-1$ ancilla qubits to measure all of the syndromes. So in total, running experiments with the surface code required $2d^2-1$ qubits. For technical reasons, we prefer $d$ to be odd, so for $d=3$ we need 17 qubits, $d=5$ we need 49 qubits, and for $d=7$ we need 97 qubits. Unfortunately our hardware doesn't have enough qubits to run $d>5$, but that's where simulation comes in! Let's see how this works.

## 5.1 CUDA-QX QEC basics

The CUDA-QX QEC library has a lot of built-in functionality for working with QEC experiments. It has some common codes built-in:

In [45]:
distance = 5
surface_code = qec.get_code("surface_code", distance=distance)

This `surface_code` object comes with a lot of useful built-in information. For example, we can list out which qubits are involved in measuring the state of the logical qubit as follows:

In [46]:
# Which physical qubits are involved in measuring Z for the logical qubit
Lz = surface_code.get_observables_z()
print(f"Z-type stabilizers:\n{Lz}")

# Which physical qubits are involved in measuring X for the logical qubit
Lx = surface_code.get_observables_x()
print(f"X-type stabilizers:\n{Lx}")

Z-type stabilizers:
[[1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 0]]
X-type stabilizers:
[[1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


We can easily tell CUDA-Q which logical state to prepare:

In [47]:
statePrep = qec.operation.prep0 # Prepare the logical state |0>

## 5.2 - GPU-accelerated decoders and simulations

Better yet, the `surface_code` object is a CUDA-Q kernel! It's not very interesting to simulate this kernel without any noise, though, so we're going to add some noise first:

In [48]:
# error probability
p = 0.001
noise = cudaq.NoiseModel()
noise.add_all_qubit_channel("x", cudaq.Depolarization2(p), 1) # add some depolarization noise to the qubits with probability p

Finally, let's run the simulation. We just have a couple of parameters to set.

Note: We add a line for how to GPU-accelerate your simulation, but the GPU simulation does a full simulation of the qubits' state. It turns out that for certain common types of noise ("Clifford" noise), more efficient methods are available without such a detailed simulation. So the CPU simulator `stim` will actually be faster.

In [ ]:
cudaq.set_target("stim") # Efficient classical CPU simulator
# cudaq.set_target("nvidia") # To run your simulation on GPU - see above discussion

# Uncomment for repeatability
cudaq.set_random_seed(13)

# Set the number of shots and the number of times we measure the stabilizers (rounds)
nShots = 10000
nRounds = distance

# Sample the surface code memory circuit with noise on each cx gate
simulated_syndromes, simulated_data = qec.sample_memory_circuit(surface_code, statePrep, nShots, nRounds, noise)

Remember how our syndromes helped us detect (and, at least in the classical case, correct) errors using "decoding"? Well, decoders are built-in to CUDA-QX QEC too.

The decoding process is a purely classical processing step - it doesn't require any additional QPU time, even if we weren't running a simulation. It can be, however, very classically intensive, so GPU-accelerated decoders are available. We'll set up a CPU decoder and a GPU-enabled decoder to explore with.

In [ ]:
# Get the detector error model for the noisy circuit.
cudaq.set_target("stim")
dem = qec.z_dem_from_memory_circuit(surface_code, statePrep, nRounds, noise)

lut_decoder = qec.get_decoder("single_error_lut", dem.detector_error_matrix) # A CPU decoder

H = surface_code.get_parity_z()
tn_decoder_gpu = qec.get_decoder("tensor_network_decoder",H,logical_obs=Lz,noise_model=noise,device="cuda")
tn_decoder_cpu = qec.get_decoder("tensor_network_decoder",H,logical_obs=Lz,noise_model=noise,device="cpu")

Now we need to process the results; we won't go into the details of this process, but we can find out the number of logical errors using our decoder.

In [ ]:
# Logical Mz each shot (use Lx if preparing in X-basis)
logical_measurements = (Lz @ simulated_data.transpose()) % 2
# only one logical qubit, so do not need the second axis
logical_measurements = logical_measurements.flatten()

# Reshape and drop the X stabilizers, keeping just the Z stabilizers (since this is prep0)
simulated_syndromes = simulated_syndromes.reshape((nShots, nRounds, -1))
simulated_syndromes = simulated_syndromes[:, :, :simulated_syndromes.shape[2] // 2]
# Now flatten to two dimensions again
simulated_syndromes = simulated_syndromes.reshape((nShots, -1))

#######
###Choose a decoder by uncommenting the line you want
#######
dr = lut_decoder.decode_batch(simulated_syndromes)
#dr = tn_decoder_gpu.decode_batch(simulated_syndromes)
#dr = tn_decoder_cpu.decode_batch(simulated_syndromes)
error_predictions = np.array([e.result for e in dr], dtype=np.uint8)
data_predictions = (dem.observables_flips_matrix @ error_predictions.T) % 2

nLogicalErrorsWithoutDecoding = np.sum(logical_measurements)
nLogicalErrorsWithDecoding = np.sum(data_predictions ^ logical_measurements)
print(
    f'Number of logical errors without decoding (out of {nShots} shots): {nLogicalErrorsWithoutDecoding}'
)
print(
    f'Number of logical errors with decoding    (out of {nShots} shots): {nLogicalErrorsWithDecoding}'
)

Number of logical errors without decoding (out of 10000 shots): 391
Number of logical errors with decoding    (out of 10000 shots): 150


> **TASK:** Experiment with the simulation and the decoder if you change the distance of the code. Does it get slow eventually? Try the different decoders and decoder devices.

In [ ]:
# Copyright 2026 IQM Quantum Computers (Daniel Bulmash)
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.